In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib scikit-learn tqdm
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q librosa soundfile pesq pystoi
!apt-get install -qq ffmpeg sox
print('✅ 环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU不可用，请在运行时类型中选择T4 GPU')

# 深度学习环境搭建

## 学习目标

- 理解Conda环境管理的作用
- 能创建、激活、管理Conda环境
- 成功安装PyTorch并验证GPU可用
- 能使用Jupyter Lab进行交互式开发
- 了解Docker的基本概念和使用方式

## 本节说明

环境搭建是深度学习中最令人头疼的环节之一。本节课的目标是让每个人都能成功运行以下代码：

```python
import torch
print(torch.cuda.is_available())  # True
```


## 1. Conda环境管理

### 1.1 为什么需要虚拟环境？

想象一下：
- 项目A需要numpy 1.21，项目B需要numpy 1.24
- 项目A需要Python 3.9，项目B需要Python 3.11

如果没有虚拟环境，这些冲突会让你抓狂。虚拟环境就是给每个项目一个独立的"工具箱"。

**类比**：虚拟环境就像实验室里的不同实验台，每个实验台有独立的仪器和试剂，互不干扰。


### 1.2 Conda基本操作

```bash
# 创建环境
conda create -n lab-training python=3.10

# 激活环境
conda activate lab-training

# 退出环境
conda deactivate

# 列出所有环境
conda env list

# 删除环境
conda env remove -n lab-training

# 从配置文件创建环境
conda env create -f environment.yml

# 导出当前环境
conda env export > environment.yml
```


### 1.3 安装包

```bash
# 安装单个包
conda install numpy

# 安装多个包
conda install numpy scipy matplotlib

# 用pip安装（conda找不到的包）
pip install pesq pystoi

# 查看已安装的包
conda list

# 更新包
conda update numpy
```

**重要规则**：
- 优先用 `conda install`，因为conda会处理二进制依赖
- 如果conda找不到，再用 `pip install`
- 不要在同一个环境中混用conda和pip安装同一个包


### 练习1：创建Conda环境

在终端中完成：

1. 创建名为 `lab-training` 的环境，Python版本3.10：
   ```bash
   conda create -n lab-training python=3.10
   ```
2. 激活环境：
   ```bash
   conda activate lab-training
   ```
3. 确认Python版本：
   ```bash
   python --version
   ```
4. 确认你在正确的环境中：
   ```bash
   which python
   ```
   输出应该包含 `lab-training` 的路径

## 2. PyTorch安装与验证

### 2.1 安装PyTorch

安装PyTorch最安全的方式是从[官网](https://pytorch.org/get-started/locally/)获取安装命令。

根据你的系统和CUDA版本选择对应的命令。以下是常见的安装命令：

```bash
# CUDA 11.8（最常用）
conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia

# CUDA 12.1
conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia

# CPU版本（没有GPU时使用）
conda install pytorch torchvision torchaudio cpuonly -c pytorch
```

**⚠️ 注意**：安装前确认服务器的CUDA版本！运行 `nvidia-smi` 查看右上角的CUDA版本。

### 2.2 验证安装

安装完成后，运行以下代码验证。这是本节课最重要的验证环节。


In [ ]:
# ====== 环境验证 ======
# 如果所有输出都正常，说明环境配置成功！

import sys
print(f'Python版本: {sys.version}')
print(f'Python路径: {sys.executable}')
print()

In [ ]:
import torch
print(f'PyTorch版本: {torch.__version__}')
print(f'CUDA可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA版本: {torch.version.cuda}')
    print(f'GPU数量: {torch.cuda.device_count()}')
    print(f'GPU名称: {torch.cuda.get_device_name(0)}')
    print(f'GPU内存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('⚠️ CUDA不可用！请检查安装。')
print()

In [ ]:
import torchaudio
print(f'torchaudio版本: {torchaudio.__version__}')
print()

In [ ]:
import numpy as np
import matplotlib
import scipy
import librosa

print(f'NumPy版本: {np.__version__}')
print(f'Matplotlib版本: {matplotlib.__version__}')
print(f'SciPy版本: {scipy.__version__}')
print(f'librosa版本: {librosa.__version__}')
print()
print('✅ 所有库导入成功！')

### 2.3 GPU快速测试

验证GPU确实可以用于计算：

In [ ]:
# GPU快速测试
if torch.cuda.is_available():
    # 在GPU上创建张量并做计算
    x = torch.randn(1000, 1000, device='cuda')
    y = torch.randn(1000, 1000, device='cuda')
    
    # 矩阵乘法
    import time
    start = time.time()
    z = torch.mm(x, y)
    torch.cuda.synchronize()  # 等待GPU计算完成
    elapsed = time.time() - start
    
    print(f'GPU矩阵乘法: 1000x1000, 耗时 {elapsed*1000:.2f} ms')
    print(f'结果张量形状: {z.shape}')
    print(f'结果张量设备: {z.device}')
    print('✅ GPU计算正常！')
else:
    print('⚠️ GPU不可用，使用CPU进行计算')
    x = torch.randn(1000, 1000)
    y = torch.randn(1000, 1000)
    z = torch.mm(x, y)
    print(f'CPU矩阵乘法结果形状: {z.shape}')

### 练习2：完成环境验证

确保以上所有验证cell的输出都是正常的。特别注意：
- `torch.cuda.is_available()` 返回 `True`
- `torchaudio` 可以正常导入
- `librosa` 可以正常导入

如果任何一步失败，请参考"环境问题排查清单"。


## 3. pip vs conda、requirements.txt

### 3.1 pip vs conda

| 特性 | conda | pip |
|------|-------|-----|
| 包来源 | Anaconda仓库 | PyPI |
| 二进制依赖 | 自动处理 | 不处理 |
| 非Python包 | 可以安装 | 不可以 |
| 速度 | 较慢 | 较快 |
| 包数量 | 较少 | 非常多 |

**规则**：优先用conda，conda找不到再用pip。


### 3.2 requirements.txt 和 environment.yml

**requirements.txt**（pip格式）：
```
numpy>=1.21
scipy>=1.7
matplotlib>=3.4
torch>=2.0
librosa>=0.9
```

**environment.yml**（conda格式）：
```yaml
name: lab-training
channels:
  - pytorch
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - numpy
  - scipy
  - matplotlib
  - pytorch>=2.0
  - torchaudio>=2.0
  - librosa
  - pip
  - pip:
    - pesq
    - pystoi
```

**environment.yml 更适合分享**——它包含了完整的环境定义，别人可以一键复现。


## 4. Docker基础

### 4.1 Docker是什么？

Docker是一种容器化技术，可以把整个环境（操作系统+依赖+代码）打包成一个"镜像"。

**类比**：
- Conda环境 = 在自己的房间里布置家具
- Docker镜像 = 把整个房间打包，搬到任何地方都能用

### 4.2 为什么用Docker？

- **环境一致性**："在我机器上能跑"的问题不存在了
- **快速部署**：一条命令启动完整环境
- **隔离性**：不同项目互不干扰


### 4.3 Docker基本使用

```bash
# 拉取镜像
docker pull pytorch/pytorch:2.1.0-cuda11.8-cudnn8-runtime

# 运行容器（带GPU支持）
docker run --gpus all -p 8888:8888 -v $(pwd):/workspace lab-training

# 参数说明：
# --gpus all     使用所有GPU
# -p 8888:8888   端口映射（Jupyter）
# -v $(pwd):/workspace  挂载当前目录到容器的/workspace
```

**你不需要会写Dockerfile**——培训者已经准备好了预配置的镜像，你只需要会用就行。


## 5. Jupyter Lab

### 5.1 启动Jupyter Lab

```bash
# 在服务器上启动
jupyter lab --no-browser --port=8888 --ip=0.0.0.0

# 参数说明：
# --no-browser    不自动打开浏览器
# --port=8888     指定端口
# --ip=0.0.0.0    允许远程访问
```

启动后，终端会显示一个URL，类似：
```
http://127.0.0.1:8888/lab?token=xxxxx
```

在本地浏览器中打开这个URL即可。


### 5.2 SSH端口转发（如果直接访问不了）

如果服务器的Jupyter端口不能直接访问，需要通过SSH端口转发：

```bash
# 在本地终端运行
ssh -L 8888:localhost:8888 username@server_ip
```

然后在本地浏览器打开 `http://localhost:8888/lab?token=xxxxx`。


### 5.3 Jupyter Lab快捷键

| 快捷键 | 功能 |
|--------|------|
| `Shift+Enter` | 运行当前cell，跳到下一个 |
| `Ctrl+Enter` | 运行当前cell，不跳转 |
| `A` | 在当前cell上方插入新cell |
| `B` | 在当前cell下方插入新cell |
| `DD` | 删除当前cell |
| `M` | 将cell切换为Markdown |
| `Y` | 将cell切换为Code |
| `Tab` | 自动补全 |
| `Shift+Tab` | 查看函数文档 |

## 6. 环境问题排查清单

### 常见问题与解决方案

| 问题 | 可能原因 | 解决方案 |
|------|---------|----------|
| `conda: command not found` | Conda未安装或未加入PATH | 安装Anaconda/Miniconda，重启终端 |
| `torch.cuda.is_available() = False` | CUDA版本不匹配 | 运行 `nvidia-smi` 检查CUDA版本，重新安装对应版本 |
| `ImportError: No module named 'xxx'` | 包未安装 | `conda install xxx` 或 `pip install xxx` |
| `OSError: libcublas.so.11` | CUDA库缺失 | 安装对应的CUDA toolkit |
| Jupyter打不开 | 端口未转发 | 检查SSH端口转发配置 |
| `permission denied` | 权限不足 | 检查文件权限，`chmod +x` 或用sudo |
| conda安装太慢 | 默认源在国外 | 配置国内镜像源 |


In [ ]:
# ====== 一键环境检测脚本 ======
# 运行此cell，如果全部显示✅，说明环境配置成功

import sys
print('='*50)
print('环境检测')
print('='*50)

# Python版本
print(f'\nPython: {sys.version.split()[0]}', '✅' if sys.version_info >= (3, 10) else '⚠️ 建议升级到3.10+')

# 核心库
libs = {
    'numpy': 'np',
    'torch': 'torch',
    'torchaudio': 'torchaudio',
    'matplotlib': 'matplotlib',
    'scipy': 'scipy',
    'librosa': 'librosa',
    'soundfile': 'sf',
}

for lib_name, import_as in libs.items():
    try:
        mod = __import__(lib_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'{lib_name}: {ver} ✅')
    except ImportError:
        print(f'{lib_name}: 未安装 ❌')

# GPU
import torch
if torch.cuda.is_available():
    print(f'\nGPU: {torch.cuda.get_device_name(0)} ✅')
    print(f'CUDA: {torch.version.cuda} ✅')
    print(f'GPU内存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB ✅')
else:
    print('\nGPU: 不可用 ⚠️ (将使用CPU，训练会较慢)')

print('\n' + '='*50)
print('如果所有项都显示✅，环境配置成功！')
print('='*50)

## 本节要点

1. **Conda环境** = 独立的工具箱，一个项目一个环境，互不干扰
2. **安装PyTorch**时一定要匹配CUDA版本，用 `nvidia-smi` 先确认
3. **优先conda安装，pip补充**——conda处理二进制依赖更好
4. **Docker**是环境的终极兜底方案——一条命令启动完整环境
5. **Jupyter Lab**是交互式开发的核心工具——学会SSH端口转发
6. 环境问题不可怕——大部分是版本不匹配，参考排查清单逐项检查


---
返回目录：[README.md](../README.md)